In [46]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision.transforms import v2
import torchvision

In [47]:
# from pathlib import Path
# import shutil
# SOURCE = Path(r"C:\Users\yashs\Downloads\Cheque Verification System\Signature Data\archive (3)\signature_ds_combined")

# DEST = Path("./dataset")
# DEST.mkdir(exist_ok=True)

# for genuine_folder in SOURCE.iterdir():

#     if not genuine_folder.is_dir():
#         continue

#     # Ignore forgery folders
#     if genuine_folder.name.endswith("_forg"):
#         continue

#     person_id = genuine_folder.name

#     forged_folder = SOURCE / f"{person_id}_forg"

#     destination_person = DEST / f"person_{person_id}"

#     genuine_dest = destination_person / "genuine"
#     forged_dest = destination_person / "forged"

#     genuine_dest.mkdir(parents=True, exist_ok=True)
#     forged_dest.mkdir(parents=True, exist_ok=True)

#     # Copy genuine images
#     for img in sorted(genuine_folder.iterdir()):
#         if img.is_file():
#             shutil.copy2(img, genuine_dest / img.name)

#     # Copy forged images
#     if forged_folder.exists():
#         for img in sorted(forged_folder.iterdir()):
#             if img.is_file():
#                 shutil.copy2(img, forged_dest / img.name)

# print("Dataset restructuring complete.")

In [48]:
# from pathlib import Path
# from PIL import Image
# import numpy as np
# import pandas as pd
# from tqdm import tqdm

# DATASET = Path("./dataset")

# stats = []

# widths = []
# heights = []
# category = [".jpg", ".png", ".jpeg"]

# for person in tqdm(sorted(DATASET.iterdir())):

#     genuine = list((person/"genuine").glob("*"))
#     forged = list((person/"forged").glob("*"))

#     stats.append({
#         "writer": person.name,
#         "genuine": len(genuine),
#         "forged": len(forged)
#     })
#     for img_path in genuine + forged:

#         if img_path.suffix not in category:
#             continue

#         img = Image.open(img_path)

#         w, h = img.size

#         widths.append(w)
#         heights.append(h)

# df = pd.DataFrame(stats)

# print(df.head())

# print()

# print("Total writers :", len(df))
# print("Total genuine :", df.genuine.sum())
# print("Total forged  :", df.forged.sum())

# print()

# print("Average width :", np.mean(widths))
# print("Average height:", np.mean(heights))

# print()

# print("Min size :", min(widths), min(heights))
# print("Max size :", max(widths), max(heights))

In [49]:
# from pathlib import Path
# from PIL import Image

# DATASET = Path("./dataset")
# category = [".jpg", ".png", ".jpeg"]

# for person in DATASET.iterdir():

#     for folder in ["genuine", "forged"]:

#         for img_path in (person/folder).glob("*"):
#             if img_path.suffix not in category:
#                 continue
#             img = Image.open(img_path)

#             w,h = img.size

#             if w < 50 or h < 50:

#                 print(img_path, img.size)

In [50]:
# from collections import Counter

# counter = Counter()
# category = [".jpg", ".png", ".jpeg"]

# for person in DATASET.iterdir():

#     for folder in ["genuine","forged"]:

#         for img_path in (person/folder).glob("*"):
#             if img_path.suffix not in category:
#                 continue
#             img = Image.open(img_path)

#             counter[img.mode]+=1

# print(counter)

In [51]:
# import matplotlib.pyplot as plt

# plt.figure(figsize=(8,4))
# plt.hist(widths, bins=50)
# plt.title("Image Width Distribution")
# plt.xlabel("Width (pixels)")
# plt.ylabel("Count")
# plt.grid(alpha=0.3)
# plt.show()

# plt.figure(figsize=(8,4))
# plt.hist(heights, bins=50)
# plt.title("Image Height Distribution")
# plt.xlabel("Height (pixels)")
# plt.ylabel("Count")
# plt.grid(alpha=0.3)
# plt.show()

# aspect_ratios = [w / h for w, h in zip(widths, heights)]

# plt.figure(figsize=(8, 4))
# plt.hist(aspect_ratios, bins=50)
# plt.title("Aspect Ratio Distribution")
# plt.xlabel("Width / Height")
# plt.ylabel("Count")
# plt.grid(alpha=0.3)
# plt.show()

In [52]:
# from pathlib import Path
# from PIL import Image

# DATASET = Path("./dataset")

# bad_images = []

# for img_path in DATASET.rglob("*"):

#     if img_path.suffix.lower() not in [".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"]:
#         continue

#     try:
#         Image.open(img_path).verify()

#     except Exception:
#         bad_images.append(img_path)

# print(f"Broken Images : {len(bad_images)}")

# if bad_images:
#     print(*bad_images[:20], sep="\n")

In [53]:
import numpy as np
import torch
from PIL import Image, ImageOps
from torchvision.transforms import v2

IMAGE_SIZE = (448, 224)  # (width, height)

class TightCropToInk:
    def __init__(self, bg_threshold=245, margin=8):
        self.bg_threshold = bg_threshold
        self.margin = margin

    def __call__(self, image):
        image = image.convert("RGB")
        arr = np.array(image)
        gray = arr.mean(axis=2)

        ys, xs = np.where(gray < self.bg_threshold)

        # If nothing meaningful is found, return original
        if len(xs) == 0 or len(ys) == 0:
            return image

        x0 = max(int(xs.min()) - self.margin, 0)
        x1 = min(int(xs.max()) + self.margin + 1, image.width)
        y0 = max(int(ys.min()) - self.margin, 0)
        y1 = min(int(ys.max()) + self.margin + 1, image.height)

        return image.crop((x0, y0, x1, y1))

In [54]:
class LetterboxResize:
    def __init__(self, target_size=(448, 224), fill=(255, 255, 255)):
        self.target_w = target_size[0]
        self.target_h = target_size[1]
        self.fill = fill

    def __call__(self, image):
        image = image.convert("RGB")
        orig_w, orig_h = image.size

        scale = min(self.target_w / orig_w, self.target_h / orig_h)

        new_w = max(1, int(round(orig_w * scale)))
        new_h = max(1, int(round(orig_h * scale)))

        image = image.resize((new_w, new_h), Image.Resampling.LANCZOS)

        pad_left = (self.target_w - new_w) // 2
        pad_right = self.target_w - new_w - pad_left

        pad_top = (self.target_h - new_h) // 2
        pad_bottom = self.target_h - new_h - pad_top

        image = ImageOps.expand(
            image,
            border=(pad_left, pad_top, pad_right, pad_bottom),
            fill=self.fill
        )

        return image

In [55]:
# DATASET = Path("./dataset")

# transform = LetterboxResize(target_size=(448, 224))

# all_images = []

# for person in DATASET.iterdir():

#     genuine_dir = person / "genuine"
#     forged_dir = person / "forged"

#     for img in genuine_dir.glob("*"):
#         all_images.append((img, "Genuine"))

#     for img in forged_dir.glob("*"):
#         all_images.append((img, "Forged"))

# samples = random.sample(all_images, 15)

# fig, axes = plt.subplots(15, 2, figsize=(12, 30))

# for row, (img_path, label) in enumerate(samples):

#     original = Image.open(img_path)

#     transformed = transform(original)

#     axes[row, 0].imshow(original)
#     axes[row, 0].set_title(
#         f"Original\n{label}\n{original.size}",
#         fontsize=10
#     )
#     axes[row, 0].axis("off")

#     axes[row, 1].imshow(transformed)
#     axes[row, 1].set_title(
#         f"Letterboxed\n{transformed.size}",
#         fontsize=10
#     )
#     axes[row, 1].axis("off")

# plt.tight_layout()
# plt.show()

In [56]:
from pathlib import Path
from sklearn.model_selection import train_test_split

DATASET = Path("./dataset")

writers = sorted([
    p.name
    for p in DATASET.iterdir()
    if p.is_dir()
])

print(f"Total Writers : {len(writers)}")

train_writers, test_writers = train_test_split(
    writers,
    test_size=0.15,
    random_state=42,
    shuffle=True
)
print(f"Train Writers : {len(train_writers)}")

Total Writers : 1487
Train Writers : 1263


In [57]:
train_writers, val_writers = train_test_split(
    train_writers,
    test_size=0.17647,     # ≈15% of total dataset
    random_state=42,
    shuffle=True
)
print(f"Train Writers : {len(train_writers)}")
print(f"Validation    : {len(val_writers)}")
print(f"Test          : {len(test_writers)}")

Train Writers : 1040
Validation    : 223
Test          : 224


In [58]:
train_writers = set(train_writers)      ## Sanity checks
val_writers = set(val_writers)
test_writers = set(test_writers)

assert train_writers.isdisjoint(val_writers)
assert train_writers.isdisjoint(test_writers)
assert val_writers.isdisjoint(test_writers)

print("No overlap found.")

No overlap found.


In [59]:
# import json

# splits = {
#     "train": sorted(train_writers),
#     "val": sorted(val_writers),
#     "test": sorted(test_writers)
# }

# with open("writer_split.json", "w") as f:
#     json.dump(splits, f, indent=4)

# print("writer_split.json saved.")

In [60]:
## Loading the Model

import torch
import torch.nn as nn
from torchvision import models

NUM_CLASSES = len(train_writers)

model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
in_features = model.fc.in_features

model.fc = nn.Sequential(
    nn.Linear(in_features, 1024),
    nn.ReLU(inplace=True),
    nn.Dropout(0.3),
    nn.Linear(1024, NUM_CLASSES)
)

In [61]:
for param in model.parameters():
    param.requires_grad = False

for param in model.fc.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())

print(f"Trainable Parameters : {trainable:,}")
print(f"Total Parameters     : {total:,}")

Trainable Parameters : 3,164,176
Total Parameters     : 26,672,208


In [62]:
train_writers = sorted(train_writers)

writer_to_label = {
    writer: idx
    for idx, writer in enumerate(train_writers)
}

label_to_writer = {
    idx: writer
    for writer, idx in writer_to_label.items()
}

In [63]:
from torch.utils.data import Dataset
from PIL import Image

from torch.utils.data import Dataset
from pathlib import Path
from PIL import Image

IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

class SignatureClassificationDataset(Dataset):
    def __init__(self, root_dir, writers, transform=None):
        self.root = Path(root_dir)
        self.transform = transform
        self.samples = []

        for writer in sorted(writers):
            if writer not in writer_to_label:
                continue

            label = writer_to_label[writer]
            genuine_dir = self.root / writer / "genuine"

            for img_path in sorted(genuine_dir.glob("*")):
                if img_path.suffix.lower() not in IMAGE_EXTS:
                    continue
                self.samples.append((img_path, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

In [64]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

train_transform = v2.Compose([
    TightCropToInk(bg_threshold=245, margin=8),
    v2.RandomApply([
        v2.RandomRotation(degrees=5, fill=(255, 255, 255))
    ], p=0.5),
    v2.RandomApply([
        v2.RandomAffine(
            degrees=0,
            translate=(0.02, 0.02),
            scale=(0.95, 1.05),
            shear=3,
            fill=(255, 255, 255)
        )
    ], p=0.5),
    LetterboxResize(target_size=IMAGE_SIZE, fill=(255, 255, 255)),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

In [65]:
val_transform = v2.Compose([
    TightCropToInk(bg_threshold=245, margin=8),
    LetterboxResize(target_size=IMAGE_SIZE, fill=(255, 255, 255)),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

In [66]:
from torch.utils.data import DataLoader

BATCH_SIZE = 32
NUM_WORKERS = 12
PIN_MEMORY = torch.cuda.is_available()

train_dataset = SignatureClassificationDataset(
    DATASET,
    train_writers,
    transform=train_transform
)

val_dataset = SignatureClassificationDataset(
    DATASET,
    val_writers,
    transform=val_transform
)

test_dataset = SignatureClassificationDataset(
    DATASET,
    test_writers,
    transform=val_transform
)

In [67]:
test_dataset = SignatureClassificationDataset(
    DATASET,
    test_writers,
    transform=val_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=PIN_MEMORY,
    persistent_workers=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=PIN_MEMORY,
    persistent_workers=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=PIN_MEMORY,
    persistent_workers=False
)

In [68]:
from pathlib import Path
from datetime import datetime
import logging
import json
import torch
import torch.nn as nn
from tqdm import tqdm


def create_run_logger(base_dir="runs", run_name="resnet50_signature"):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    run_dir = Path(base_dir) / f"{run_name}_{timestamp}"
    run_dir.mkdir(parents=True, exist_ok=True)

    logger = logging.getLogger(f"{run_name}_{timestamp}")
    logger.setLevel(logging.INFO)
    logger.propagate = False
    logger.handlers.clear()

    log_file = run_dir / "train.log"

    formatter = logging.Formatter(
        "%(asctime)s | %(levelname)s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S"
    )

    file_handler = logging.FileHandler(log_file, mode="w", encoding="utf-8")
    file_handler.setFormatter(formatter)
    file_handler.setLevel(logging.INFO)

    stream_handler = logging.StreamHandler()
    stream_handler.setFormatter(formatter)
    stream_handler.setLevel(logging.INFO)

    logger.addHandler(file_handler)
    logger.addHandler(stream_handler)

    return logger, run_dir


def freeze_batchnorm_layers(model):
    """
    Keep BatchNorm layers in eval mode so their running stats do not update.
    Useful when the backbone is frozen.
    """
    for module in model.modules():
        if isinstance(module, nn.BatchNorm2d):
            module.eval()


def train_one_epoch(model, loader, optimizer, criterion, device, freeze_bn=True):
    model.train()
    if freeze_bn:
        freeze_batchnorm_layers(model)

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(loader, desc="Train", leave=False):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * labels.size(0)

        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total

    return epoch_loss, epoch_acc


@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(loader, desc="Val", leave=False):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        outputs = model(images)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * labels.size(0)

        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total

    return epoch_loss, epoch_acc


def save_checkpoint(path, model, optimizer, epoch, best_val_acc, extra=None):
    checkpoint = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "best_val_acc": best_val_acc,
    }
    if extra is not None:
        checkpoint["extra"] = extra

    torch.save(checkpoint, path)


def train_model(
    model,
    train_loader,
    val_loader,
    optimizer,
    criterion,
    device,
    epochs=10,
    scheduler=None,
    freeze_bn=True,
    run_name="resnet50_signature",
):
    logger, run_dir = create_run_logger(run_name=run_name)

    config = {
        "epochs": epochs,
        "freeze_bn": freeze_bn,
        "device": str(device),
        "train_batches": len(train_loader),
        "val_batches": len(val_loader),
    }

    with open(run_dir / "config.json", "w", encoding="utf-8") as f:
        json.dump(config, f, indent=4)

    logger.info("Run directory: %s", run_dir)
    logger.info("Starting training")
    logger.info("Config: %s", config)

    best_val_acc = 0.0
    best_epoch = -1

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            criterion=criterion,
            device=device,
            freeze_bn=freeze_bn,
        )

        val_loss, val_acc = validate(
            model=model,
            loader=val_loader,
            criterion=criterion,
            device=device,
        )

        if scheduler is not None:
            scheduler.step()

        current_lr = optimizer.param_groups[0]["lr"]

        logger.info(
            "Epoch %d/%d | LR %.6f | Train Loss %.4f | Train Acc %.4f | Val Loss %.4f | Val Acc %.4f",
            epoch, epochs, current_lr, train_loss, train_acc, val_loss, val_acc
        )

        last_ckpt_path = run_dir / "last_checkpoint.pth"
        save_checkpoint(
            last_ckpt_path,
            model=model,
            optimizer=optimizer,
            epoch=epoch,
            best_val_acc=best_val_acc
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = epoch

            best_ckpt_path = run_dir / "best_checkpoint.pth"
            save_checkpoint(
                best_ckpt_path,
                model=model,
                optimizer=optimizer,
                epoch=epoch,
                best_val_acc=best_val_acc
            )

            logger.info(
                "Saved new best checkpoint at epoch %d with val acc %.4f",
                epoch, val_acc
            )

    logger.info("Training complete")
    logger.info("Best Val Acc: %.4f at epoch %d", best_val_acc, best_epoch)

    return {
        "run_dir": str(run_dir),
        "best_val_acc": best_val_acc,
        "best_epoch": best_epoch,
    }

In [69]:
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model = model.to(device)

# # If you are only training the head:
# for param in model.parameters():
#     param.requires_grad = False

# for param in model.fc.parameters():
#     param.requires_grad = True

# optimizer = torch.optim.AdamW(
#     model.fc.parameters(),
#     lr=1e-3,
#     weight_decay=1e-4
# )

# criterion = nn.CrossEntropyLoss()

# scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
#     optimizer,
#     T_max=10
# )

# results = train_model(
#     model=model,
#     train_loader=train_loader,
#     val_loader=val_loader,
#     optimizer=optimizer,
#     criterion=criterion,
#     device=device,
#     epochs=10,
#     scheduler=scheduler,
#     freeze_bn=True,
#     run_name="signature_resnet50_stage1"
# )

# print(results)